# 02 · Motor de texto con BETO

**Fase 3 — El núcleo del aporte. Corre en Google Colab con GPU.**

Usa las particiones anonimizadas del repo (`data/anonimizado/particiones/`): **val y test son 100% reales**.

**Pasos en Colab:** 1) Entorno de ejecución → Cambiar tipo → **GPU (T4)** *(hazlo PRIMERO)* · 2) **Ejecutar todo**.
No hace falta subir ni descomentar nada: el notebook instala y clona solo.

## 0. Preparación (solo en Colab)

In [10]:
# Prepara el entorno en Colab (en local no hace nada).
import os, sys, subprocess
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate'])
print('Entorno: Colab' if EN_COLAB else 'Entorno: local')

Entorno: Colab


In [11]:
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Sin GPU: activa GPU en Colab (Entorno de ejecución → Cambiar tipo).')

GPU disponible: True
GPU: Tesla T4


## 1. Cargar las particiones (val y test = reales)

In [12]:
from pathlib import Path
import os, sys, subprocess
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

# En Colab: si el repo no está (p. ej. al cambiar a GPU la máquina se reinicia), se clona aquí mismo.
if 'google.colab' in sys.modules and not Path('Modelotest').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Ortunex1/Modelotest.git'], check=True)

def buscar(nombre):
    for p in [Path('Modelotest/data/anonimizado/particiones') / nombre,   # Colab
              Path('data/anonimizado/particiones') / nombre,
              Path.cwd().parent / 'data' / 'anonimizado' / 'particiones' / nombre,  # local
              Path(nombre)]:
        if p.exists():
            return p
    return None

p_tr, p_va, p_te = buscar('corpus_train.csv'), buscar('corpus_val.csv'), buscar('corpus_test.csv')
if p_tr is None or p_va is None:
    raise FileNotFoundError('No se encontraron corpus_train.csv / corpus_val.csv. '
                            'En Colab revisa que el git clone haya funcionado.')

train = pd.read_csv(p_tr)[['texto', 'etiqueta']].dropna()
val = pd.read_csv(p_va)[['texto', 'etiqueta']].dropna()
test = pd.read_csv(p_te)[['texto', 'etiqueta']].dropna() if p_te is not None else None
print('Particiones REALES cargadas ✔ desde', p_tr.parent)
print(f'train={len(train)}  val={len(val)}  test={len(test) if test is not None else 0}')
print('Train:', train['etiqueta'].value_counts().to_dict())
print('Val  :', val['etiqueta'].value_counts().to_dict())

Particiones REALES cargadas ✔ desde Modelotest/data/anonimizado/particiones
train=1067  val=89  test=89
Train: {'legitimo': 550, 'fraudulento': 517}
Val  : {'legitimo': 50, 'fraudulento': 39}


## 2. Línea base clásica en español (el rival)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score

base = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LogisticRegression(max_iter=1000, class_weight='balanced'),
)
base.fit(train['texto'], train['etiqueta'])
pred_base = base.predict(val['texto'])
f1_base = f1_score(val['etiqueta'], pred_base, pos_label='fraudulento', zero_division=0)
print(f'Baseline clásico — F1 (fraudulento) = {f1_base:.3f}')
print(classification_report(val['etiqueta'], pred_base, digits=3, zero_division=0))

Baseline clásico — F1 (fraudulento) = 0.900
              precision    recall  f1-score   support

 fraudulento      0.878     0.923     0.900        39
    legitimo      0.938     0.900     0.918        50

    accuracy                          0.910        89
   macro avg      0.908     0.912     0.909        89
weighted avg      0.911     0.910     0.910        89



## 3. Ajuste fino de BETO

In [14]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from torch.utils.data import Dataset
from collections import Counter

CKPT = 'dccuchile/bert-base-spanish-wwm-cased'  # BETO
ETIQUETAS = ['legitimo', 'fraudulento']
lab2id = {l: i for i, l in enumerate(ETIQUETAS)}

tok = AutoTokenizer.from_pretrained(CKPT)

class DatasetTexto(Dataset):
    def __init__(self, textos, etiquetas):
        self.enc = tok(list(textos), truncation=True, padding=True, max_length=96)
        self.labels = [lab2id[e] for e in etiquetas]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item

ds_tr = DatasetTexto(train['texto'], train['etiqueta'])
ds_val = DatasetTexto(val['texto'], val['etiqueta'])

In [15]:
# Pesos por clase para la pérdida (ponderar en vez de descartar)
cnt = Counter(train['etiqueta'])
total = sum(cnt.values())
pesos = torch.tensor([total / (2 * cnt[l]) for l in ETIQUETAS], dtype=torch.float)
print('Conteo por clase:', dict(cnt))

class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=pesos.to(model.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

Conteo por clase: {'legitimo': 550, 'fraudulento': 517}


In [16]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def metricas(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    labels = pred.label_ids
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1],
                                                  average=None, zero_division=0)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_fraudulento': f1[1],
            'recall_fraudulento': r[1],
            'precision_fraudulento': p[1]}

In [17]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=2, id2label={i: l for l, i in lab2id.items()}, label2id=lab2id)

args = TrainingArguments(
    output_dir='beto_ckpt',
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',        # en transformers viejos: evaluation_strategy
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1_fraudulento',
    seed=SEED,
    logging_steps=20,
    report_to='none',
)

trainer = TrainerPonderado(model=modelo, args=args,
                           train_dataset=ds_tr, eval_dataset=ds_val,
                           compute_metrics=metricas)
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Epoch,Training Loss,Validation Loss,Accuracy,F1 Fraudulento,Recall Fraudulento,Precision Fraudulento
1,0.097269,0.236950,0.898876,0.880000,0.846154,0.916667
2,0.019548,0.447274,0.898876,0.869565,0.769231,1.000000
3,0.000525,0.343737,0.910112,0.894737,0.871795,0.918919
4,0.000451,0.366249,0.910112,0.894737,0.871795,0.918919


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=268, training_loss=0.04485797244228367, metrics={'train_runtime': 174.1838, 'train_samples_per_second': 24.503, 'train_steps_per_second': 1.539, 'total_flos': 120630252217200.0, 'train_loss': 0.04485797244228367, 'epoch': 4.0})

## 4. Evaluación y comparación con la línea base

In [18]:
res = trainer.evaluate()
f1_beto = res['eval_f1_fraudulento']
print(f'BETO     — F1 (fraudulento) = {f1_beto:.3f}')
print(f'Baseline — F1 (fraudulento) = {f1_base:.3f}')
print('\nBETO gana al baseline' if f1_beto > f1_base else '\nRevisar: el baseline no fue superado')
print('(El veredicto formal se hace con McNemar sobre el TEST en 03_evaluacion.)')

Training Loss,Validation Loss,Epoch,Accuracy,F1 Fraudulento,Recall Fraudulento,Precision Fraudulento
0.000451,0.343737,4,0.910112,0.894737,0.871795,0.918919


BETO     — F1 (fraudulento) = 0.895
Baseline — F1 (fraudulento) = 0.900

Revisar: el baseline no fue superado
(El veredicto formal se hace con McNemar sobre el TEST en 03_evaluacion.)


In [19]:
# Guardar el modelo ajustado y las predicciones de validación
trainer.save_model('beto_final')
tok.save_pretrained('beto_final')

logits = trainer.predict(ds_val).predictions
logits = logits[0] if isinstance(logits, tuple) else logits
pred_beto = [ETIQUETAS[i] for i in np.argmax(logits, axis=1)]
pd.DataFrame({'texto': val['texto'].values, 'y_real': val['etiqueta'].values,
              'pred_beto': pred_beto, 'pred_baseline': pred_base}).to_csv(
    '02_texto_val_pred.csv', index=False)
print('Modelo en beto_final/ y predicciones en 02_texto_val_pred.csv ✔')
print('⚠️ En Colab: descarga la carpeta beto_final o cópiala a Google Drive antes de cerrar.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo en beto_final/ y predicciones en 02_texto_val_pred.csv ✔
⚠️ En Colab: descarga la carpeta beto_final o cópiala a Google Drive antes de cerrar.


## Qué sigue

- El **test** sigue sin abrirse: se usa en `03_evaluacion.ipynb` (McNemar entre BETO y baseline, criterio F1 ≥ 0.90).
- Si BETO sobreajusta (F1 alto en train, bajo en val): reduce épocas, sube el peso de la clase minoritaria o agrega más datos reales.